In [0]:
dbutils.library.restartPython()

In [0]:
spark.sql("""
DROP TABLE IF EXISTS retaildataplatform.silver.customers
""")

In [0]:
import pyspark.sql.functions  as F

from pyspark.sql.functions import expr, from_unixtime
from common_utils.scd_functions import scd1
from common_utils.logging import get_logger

logger = get_logger("customers_silver")


logger.info("Reading customer data from Bronze")
df = spark.read.table("retaildataplatform.bronze.sqlserver_customers")
logger.info("Bronze customer data successfully loaded")

def transform_customers (df):
    df = df.withColumn("customer_name", F.lower(df["customer_name"]))\
        .withColumn("name_parts", F.split(F.col("customer_name"), ","))\
        .withColumn("first_name",F.trim(F.get(F.col("name_parts"), 1)))\
        .withColumn("last_name",F.trim(F.get(F.col("name_parts"), 0)))\
        .withColumn("city", F.upper(F.col("city")))\
        .withColumn("postcode",F.regexp_replace(F.col("postcode"), r"\.0$", ""))\
        .withColumn("valid_from", from_unixtime(expr("try_cast(valid_from as bigint)"))) \
        .withColumn("valid_to", from_unixtime(expr("try_cast(valid_to as bigint)")))\
        .withColumn("country", F.lit("USA"))


    df = df.drop("name_parts")\
        .drop("customer_name")\
        .drop("file_path") 
    
    return df.select(
           "customer_id",
            "first_name",
            "last_name",
            "tax_id",
            "tax_code",
            "state",
            "city",
            "postcode",
            "street",
            "country",
            "number",
            "unit",
            "region",
            "district",
            "lon",
            "lat",
            "ship_to_address",
            "valid_from",
            "valid_to",
            "units_purchased",
            "loyalty_segment",
            "last_update_ts")
    

cleaned_customer = transform_customers(df)
logger.info("Customer transformation completed")
cleaned_customer.display()


scd1(
    spark,
    cleaned_customer,
    "customer_id",
    "retaildataplatform.silver.customers"
)
logger.info("Customer data successfully processed into Silver")
 
            
            




In [0]:
cleaned_customer = transform_customers(df)

cleaned_customer.printSchema()

In [0]:
import common_utils.scd_functions as scd_functions
import importlib

importlib.reload(scd_functions)

scd1 = scd_functions.scd1